# Treino do Modelo de Deteção de Malware

## Objetivo

Preparar os dados rotulados, reservar uma partição de teste que não será usada durante o treino, comparar classificadores por validação cruzada e treinar o modelo final **Extra Trees**. Este notebook é apenas a etapa de treino; a avaliação final está em `teste_modelo.ipynb`.

**Como apresentar:** explique primeiro a separação treino/teste, depois a validação cruzada feita apenas no treino e, por fim, o modelo guardado para avaliação independente.

## 1. Bibliotecas e configuração

Reutilizamos as funções de preparação e os pipelines do projeto, para que este notebook mantenha o mesmo processo de `src/train.py`. A semente fixa permite repetir a mesma divisão de dados.

In [9]:
from pathlib import Path
import sys

import joblib
import pandas as pd
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path("..").resolve()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from app_paths import DATA, MODEL_OUTPUT, RESULTS, ensure_output_dirs
from data_preprocessing import TARGET, build_model, compare_models, load_data, prepare_data

RANDOM_STATE = 42
TEST_SIZE = 0.20
ensure_output_dirs()
print("Raiz do projeto:", PROJECT_ROOT)
print("Dataset:", DATA)

Raiz do projeto: C:\Users\istec\Documents\malware-detection-ml-final-clean\malware-detection-ml-final-clean
Dataset: C:\Users\istec\Documents\malware-detection-ml-final-clean\malware-detection-ml-final-clean\data\Malware_and_benign_recognition.csv


## 2. Carregamento e preparação dos dados

Cada linha representa um executável; `Malicious` é a label (0 = benigno, 1 = malware). `prepare_data` remove identificadores e target das entradas, converte os atributos para numéricos e cria as features derivadas documentadas no projeto. O tratamento aprendido dos dados — imputação, clipping e escala — permanece dentro do pipeline, para ser ajustado somente com dados de treino.

## 3. Separação treino/teste

Reservamos 20% para o teste final e usamos estratificação para preservar a proporção entre as classes. O teste é guardado em ficheiro para ser carregado pelo notebook de avaliação, sem participar na validação cruzada nem no ajuste do classificador.

In [10]:
df = load_data(DATA)
X, y = prepare_data(df)

print(f"Dataset preparado: {len(X)} amostras e {X.shape[1]} features.")
print("Distribuição original das classes (0 = benigno, 1 = malware):")
display(y.value_counts().sort_index().rename("Amostras").to_frame())

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

train_path = RESULTS / "malware_train_split.csv"
test_path = RESULTS / "malware_test_split.csv"
pd.concat([X_train, y_train.rename(TARGET)], axis=1).to_csv(train_path, index=False)
pd.concat([X_test, y_test.rename(TARGET)], axis=1).to_csv(test_path, index=False)

print(f"\nTreino: {len(X_train)} amostras ({len(X_train) / len(X):.0%})")
print(f"Teste reservado: {len(X_test)} amostras ({len(X_test) / len(X):.0%})")
print("Distribuição das classes no treino:")
display(y_train.value_counts(normalize=True).sort_index().mul(100).round(1).rename("Percentagem").to_frame())
print("Distribuição das classes no teste:")
display(y_test.value_counts(normalize=True).sort_index().mul(100).round(1).rename("Percentagem").to_frame())
print("\nCSV de treino:", train_path)
print("CSV de teste (não usado nesta etapa):", test_path)

Dataset preparado: 1156 amostras e 28 features.
Distribuição original das classes (0 = benigno, 1 = malware):


,Amostras
Malicious,
0,583
1,573



Treino: 924 amostras (80%)
Teste reservado: 232 amostras (20%)
Distribuição das classes no treino:


,Percentagem
Malicious,
0,50.4
1,49.6


Distribuição das classes no teste:


,Percentagem
Malicious,
0,50.4
1,49.6



CSV de treino: C:\Users\istec\Documents\malware-detection-ml-final-clean\malware-detection-ml-final-clean\results\malware_train_split.csv
CSV de teste (não usado nesta etapa): C:\Users\istec\Documents\malware-detection-ml-final-clean\malware-detection-ml-final-clean\results\malware_test_split.csv


## 4. Comparação e treino

Comparamos Logistic Regression, Random Forest, Gradient Boosting e Extra Trees com validação cruzada estratificada de 5 folds, usando apenas a partição de treino. As métricas médias ajudam a comparar estabilidade e desempenho sem consultar o teste final.

O Extra Trees é o classificador final adotado pelo projeto. O pipeline inclui o pré-processamento e o classificador; ao chamar `fit`, os passos de preparação são ajustados exclusivamente nos dados de treino.

In [11]:
train_df = pd.read_csv(train_path)
y_train_only = train_df.pop(TARGET)
X_train_only = train_df

cv_results = compare_models(X_train_only, y_train_only, cv_folds=5)
cv_path = RESULTS / "model_comparison_cv.csv"
cv_results.to_csv(cv_path, index=False)

print("Resultados médios de validação cruzada (5 folds), apenas no treino:")
display(cv_results.style.format({
    "Accuracy": "{:.2%}",
    "Precision": "{:.2%}",
    "Recall": "{:.2%}",
    "F1": "{:.2%}",
    "ROC-AUC": "{:.2%}",
}))
print("Tabela guardada em:", cv_path)

Resultados médios de validação cruzada (5 folds), apenas no treino:


,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,Gradient Boosting,99.03%,98.49%,99.56%,99.02%,99.91%
1,Extra Trees,99.46%,99.13%,99.78%,99.45%,99.84%
2,Random Forest,99.35%,98.71%,100.00%,99.35%,99.82%
3,Logistic Regression,98.27%,98.46%,98.04%,98.24%,99.57%


Tabela guardada em: C:\Users\istec\Documents\malware-detection-ml-final-clean\malware-detection-ml-final-clean\results\model_comparison_cv.csv


In [12]:
model = build_model("Extra Trees")
model.fit(X_train_only, y_train_only)
joblib.dump(model, MODEL_OUTPUT)

print("Modelo final: Extra Trees")
print(f"Treinado com {len(X_train_only)} amostras e {X_train_only.shape[1]} features.")
print("Pipeline completo guardado em:", MODEL_OUTPUT)
print("O conjunto de teste não foi utilizado no treino.")

Modelo final: Extra Trees
Treinado com 924 amostras e 28 features.
Pipeline completo guardado em: C:\Users\istec\Documents\malware-detection-ml-final-clean\malware-detection-ml-final-clean\models\modelo_extra_trees.joblib
O conjunto de teste não foi utilizado no treino.


## Resultado do treino

O artefacto guardado contém o pipeline completo, incluindo preparação e classificador. O notebook seguinte carrega este ficheiro e calcula as métricas apenas na partição de teste guardada. A validação cruzada é uma estimativa interna ao treino; não substitui essa avaliação final.